# How the population ridge decoder works

This notebook demonstrates how the Ridge population decoder works, using a single
**example session** for the neural decoding and a small cross-session sweep for the
underlying kinematic geometry.

All neural decoding here uses the **cut-treadmill** trials (the steady-state locomotion
portion, with the initial acceleration ramp excluded), matching the main analyses.

### The RS / OF / depth geometry

In the virtual corridor the relationship between running speed (RS), optic flow (OF) and
corridor depth is

$$OF = C \cdot \frac{RS}{\text{depth}} \quad\Longrightarrow\quad \log(\text{depth}) = \log(C) + \log(RS) - \log(OF).$$

So in log space **depth is the RS/OF *ratio*** ($\log RS - \log OF$). The axis **orthogonal**
to depth is the RS/OF *product*:

$$\log(RS \cdot OF) = \log(RS) + \log(OF),$$

which we decode as a separate target, `rsof_product_stim` ("RS×OF"). Depth and RS×OF
together span the (log RS, log OF) plane.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy.stats import pearsonr
from scipy.interpolate import interp1d
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from tqdm.notebook import tqdm

import flexiznam as flz
from cottage_analysis.analysis import population_ridge_decoder, treadmill, spheres
from cottage_analysis.summary_analysis import get_session_list, load_project_subsets

# Style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

ALPHAS = [0.01, 0.1, 1, 10, 100, 1000]

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording, recording was wrongly started"
}


def safe_log(arr):
    """Safely apply log-transform by setting non-positive values to NaN."""
    target = np.array(arr, dtype=float)
    target[target <= 0] = np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.log(target)


def add_rsof_product(trials_df):
    """Add the depth-orthogonal RS x OF product column (per-frame) to a trials_df."""
    trials_df = trials_df.copy()
    def _prod(row):
        rs = np.asarray(row["RS_stim"], dtype=float)
        of = np.asarray(row["OF_stim"], dtype=float)
        n = min(len(rs), len(of))
        return rs[:n] * of[:n]
    trials_df["rsof_product_stim"] = trials_df.apply(_prod, axis=1)
    return trials_df

## 1. The kinematic ceiling: decoding depth from RS and OF alone

Before looking at neurons, we confirm the geometry above: decoding $\log(\text{depth})$
from the kinematics with linear regression should reach $R^2 \approx 1$ when **both** RS and
OF are used, but is systematically biased when only OF (or only RS) is available — because
OF-only ignores the independent variation of RS. This is the kinematic upper bound the
neural decoders are compared against.

In [ ]:
def load_session_trials(sess, project, flexilims_session, cut_treadmill=True):
    """Load and synchronize treadmill or spheres recording for a session."""
    if ("PZAH6.4b" in sess) or ("PZAG3.4f" in sess):
        photodiode_protocol = 2
    else:
        photodiode_protocol = 5

    exp_session = flz.get_entity(
        datatype="session", name=sess, flexilims_session=flexilims_session
    )
    recordings = flz.get_entities(
        datatype="recording",
        origin_id=exp_session["id"],
        flexilims_session=flexilims_session,
    )
    is_treadmill = False
    if "protocol" in recordings.columns:
        is_treadmill = (
            recordings["protocol"].str.contains("SpheresTubeMotor|Treadmill", na=False).any()
        )
    if not is_treadmill:
        is_treadmill = (
            recordings["name"].str.contains("SpheresTubeMotor|Treadmill", na=False).any()
        )

    filter_datasets = {"anatomical_only": 3, "ast_neuropil": False}
    annotated_filter = dict(filter_datasets)
    annotated_filter["annotated"] = True

    def _do_sync(filt):
        if is_treadmill:
            acc_time = 0.13 if cut_treadmill else None
            return treadmill.sync_all_recordings(
                session_name=sess, flexilims_session=flexilims_session, project=project,
                filter_datasets=filt, recording_type="two_photon",
                photodiode_protocol=photodiode_protocol, return_volumes=True,
                acceleration_time=acc_time, cut_trial_end=None, trial_duration=None,
            )
        else:
            return spheres.sync_all_recordings(
                session_name=sess, flexilims_session=flexilims_session, project=project,
                filter_datasets=filt, recording_type="two_photon",
                protocol_base="SpheresPermTubeReward",
                photodiode_protocol=photodiode_protocol, return_volumes=True,
            )

    try:
        _, trials_df = _do_sync(annotated_filter)
    except FileNotFoundError:
        _, trials_df = _do_sync(filter_datasets)
    return trials_df


def decode_depth_from_features(trials_df, feature_cols, k_folds=5, random_state=42, return_details=False):
    """k-fold (trial-level) linear regression decoding depth from log kinematics."""
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=random_state)
    trial_indices = np.arange(len(trials_df))
    y_test_all, y_pred_all, details = [], [], []

    for train_idx, test_idx in kf.split(trial_indices):
        train_trials = trials_df.iloc[train_idx]
        test_trials = trials_df.iloc[test_idx]

        X_train_list, y_train_list = [], []
        for _, row in train_trials.iterrows():
            n_frames = len(row["RS_stim"])
            y_train_list.append(np.repeat(row["depth"], n_frames))
            X_train_list.append(np.column_stack([row[col] for col in feature_cols]))
        X_train_log = safe_log(np.vstack(X_train_list))
        y_train_log = safe_log(np.hstack(y_train_list))

        valid_train = ~np.isnan(y_train_log) & ~np.any(np.isnan(X_train_log), axis=1)
        X_train_log, y_train_log = X_train_log[valid_train], y_train_log[valid_train]
        if len(y_train_log) == 0:
            continue

        model = LinearRegression()
        model.fit(X_train_log, y_train_log)

        for _, row in test_trials.iterrows():
            n_frames = len(row["RS_stim"])
            feats = np.column_stack([row[col] for col in feature_cols])
            X_test_log = safe_log(feats)
            y_test_log = safe_log(np.repeat(row["depth"], n_frames))
            rs_log = safe_log(np.column_stack([row["RS_stim"]]))

            valid_test = ~np.isnan(y_test_log) & ~np.any(np.isnan(X_test_log), axis=1)
            if return_details:
                valid_test = valid_test & ~np.any(np.isnan(rs_log), axis=1)
            if np.sum(valid_test) > 0:
                pred = model.predict(X_test_log[valid_test])
                y_pred_all.append(pred)
                y_test_all.append(y_test_log[valid_test])
                if return_details:
                    for p, t, r in zip(pred, y_test_log[valid_test], rs_log[valid_test].flatten()):
                        details.append({
                            "error": t - p, "abs_error": abs(t - p),
                            "true_log_depth": t, "true_depth": np.exp(t),
                            "rs_log": r, "rs": np.exp(r),
                        })

    if len(y_test_all) == 0:
        return (np.nan, []) if return_details else np.nan
    y_test_all = np.concatenate(y_test_all)
    y_pred_all = np.concatenate(y_pred_all)
    r2 = r2_score(y_test_all, y_pred_all)
    return (r2, details) if return_details else r2

In [ ]:
projects = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]
results, all_errors = [], []
cut_treadmill = True  # use cut-treadmill trials throughout

for project in projects:
    flexilims_session = flz.get_flexilims_session(project_id=project)
    df_subsets = load_project_subsets(
        project, session_to_exclude=SESSIONS_TO_EXCLUDE.keys(), cut_treadmill=cut_treadmill
    )
    sessions = sorted(df_subsets["session"].unique())
    project_sessions = flz.get_entities("session", flexilims_session=flexilims_session)

    print(f"\nProcessing {len(sessions)} sessions in project {project}...")
    for sess in tqdm(sessions):
        if sess in SESSIONS_TO_EXCLUDE or sess not in project_sessions.index:
            continue
        nominal_depth = project_sessions.loc[sess, "nominal_depth"]
        if isinstance(nominal_depth, (list, np.ndarray, pd.Series)):
            nominal_depth = np.mean(nominal_depth)
        try:
            trials_df = load_session_trials(sess, project, flexilims_session, cut_treadmill=cut_treadmill)
            trials_df = trials_df[trials_df.closed_loop == 1].copy()
            if len(trials_df) == 0:
                continue
            r2_of, of_details = decode_depth_from_features(trials_df, ["OF_stim"], return_details=True)
            r2_rs = decode_depth_from_features(trials_df, ["RS_stim"])
            r2_both = decode_depth_from_features(trials_df, ["RS_stim", "OF_stim"])
            results.append({"session": sess, "project": project, "nominal_depth": nominal_depth,
                            "r2_OF": r2_of, "r2_RS": r2_rs, "r2_both": r2_both})
            for d in of_details:
                d["session"], d["project"] = sess, project
                all_errors.append(d)
        except Exception as e:
            print(f"Error processing session {sess}: {e}")

results_df = pd.DataFrame(results)
errors_df = pd.DataFrame(all_errors)
print(f"\n{len(results_df)} sessions decoded.")

In [ ]:
# Decoding accuracy: OF only vs RS only vs both
df_melt = pd.melt(results_df, id_vars=["session", "project", "nominal_depth"],
                  value_vars=["r2_OF", "r2_RS", "r2_both"], var_name="decoder", value_name="r2")
df_melt["decoder"] = df_melt["decoder"].map({
    "r2_OF": "Optic Flow (OF) only", "r2_RS": "Running Speed (RS) only", "r2_both": "Both (RS + OF)"})

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_melt, x="decoder", y="r2", palette="Set2", showfliers=False)
sns.stripplot(data=df_melt, x="decoder", y="r2", color="black", alpha=0.6, jitter=0.2, size=6)
plt.title("Decoding Depth from Treadmill Kinematics (No Neurons)")
plt.ylabel("Cross-validated $R^2$"); plt.xlabel("Decoder Type")
plt.ylim(-0.1, 1.1); sns.despine(); plt.tight_layout()

### Structure of the OF-only error

Because OF-only ignores RS, its depth-prediction error is structured: it correlates with
both the true depth and the running speed. This is exactly the RS-dependent component that
the depth-orthogonal RS×OF axis captures.

In [ ]:
corr_depth_err, p_depth_err = pearsonr(errors_df["true_log_depth"], errors_df["error"])
corr_depth_abs, p_depth_abs = pearsonr(errors_df["true_log_depth"], errors_df["abs_error"])
corr_rs_err, p_rs_err = pearsonr(errors_df["rs_log"], errors_df["error"])
corr_rs_abs, p_rs_abs = pearsonr(errors_df["rs_log"], errors_df["abs_error"])

print("--- Correlation with Prediction Error (true - pred) ---")
print(f"True Log Depth vs Error: r = {corr_depth_err:.4f} (p = {p_depth_err:.2e})")
print(f"Log RS vs Error:         r = {corr_rs_err:.4f} (p = {p_rs_err:.2e})")
print("\n--- Correlation with Absolute Error ---")
print(f"True Log Depth vs |Error|: r = {corr_depth_abs:.4f} (p = {p_depth_abs:.2e})")
print(f"Log RS vs |Error|:         r = {corr_rs_abs:.4f} (p = {p_rs_abs:.2e})")

In [ ]:
sample_df = errors_df.sample(n=min(len(errors_df), 10000), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.regplot(data=sample_df, x="true_depth", y="error", ax=axes[0])
axes[0].set_title("OF-Only Prediction Error vs True Depth")
axes[0].set_xlabel("True Corridor Depth (cm)"); axes[0].set_ylabel("Prediction Error (Log Depth Diff)")
axes[0].axhline(0, color="red", linestyle="--", alpha=0.7)
sns.regplot(data=sample_df, x="rs_log", y="error", ax=axes[1],
            scatter_kws={"alpha": 0.2, "color": "purple"}, line_kws={"color": "red"})
axes[1].set_title("OF-Only Prediction Error vs Log RS")
axes[1].set_xlabel("Log Running Speed"); axes[1].set_ylabel("Prediction Error (Log Depth Diff)")
axes[1].axhline(0, color="red", linestyle="--", alpha=0.7)
plt.tight_layout()

## 2. Neural population decoding in an example session

We now decode each target — OF, RS, depth, and the depth-orthogonal **RS×OF** — from the
population activity of a single example session, on the cut-treadmill closed-loop trials.

In [ ]:
EXAMPLE_SESSION = 'PZAG17.3a_S20250402'
project_rev = "colasa_3d-vision_revisions"
flm_sess_rev = flz.get_flexilims_session(project_id=project_rev)
example_mouse, example_session = EXAMPLE_SESSION.split('_')
print(f"Loading example session {example_session} (mouse {example_mouse})")

# Cut-treadmill trials (acceleration ramp excluded -> default acceleration_time)
_, trials_df = treadmill.sync_all_recordings(
    session_name=EXAMPLE_SESSION, project=project_rev, photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True}, recording_type="two_photon",
)

# Frame rate
suite2p_ds = flz.get_datasets(
    origin_name=EXAMPLE_SESSION, dataset_type='suite2p_rois',
    filter_datasets=dict(annotated=True), flexilims_session=flm_sess_rev, allow_multiple=False)
fs = suite2p_ds.extra_attributes['fs']

# Build the depth-orthogonal RS x OF target column
trials_df = add_rsof_product(trials_df)
print(f"Frame rate: {fs:.2f} Hz | {len(trials_df)} trials")

In [ ]:
def fit_target(target_col, decoder_func):
    """Fit a target decoder (+shuffle control) and store predictions back on trials_df."""
    common = dict(trials_df=trials_df, closed_loop=1, rolling_window=None,
                  downsample_window=None, frame_rate=fs, log_transform=True,
                  alphas=ALPHAS, k_folds=5, random_state=42)
    res = decoder_func(**common)
    res_shuffle = decoder_func(**common, shuffle_control=True)
    return res, res_shuffle


def plot_target(res, name, unit):
    """Scatter (actual vs predicted) + time-series for a fitted decoder result."""
    actual = np.concatenate([a for a in res['y_test_trials'] if a is not None])
    pred = np.concatenate([p for p in res['y_pred_trials'] if p is not None])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    lims = np.nanpercentile(actual, [1, 99])
    axes[0].plot(lims, lims, 'r--', lw=1)
    axes[0].scatter(actual, pred, alpha=0.1, s=1)
    axes[0].set_xlabel(f'Actual log({name}) ({unit})')
    axes[0].set_ylabel(f'Predicted log({name}) ({unit})')
    axes[0].set_title(f'{name} decoder: R² = {res["r2"]:.3f}, r = {res["pearson_r"]:.3f}')
    n_show = min(5000, len(actual))
    axes[1].plot(actual[:n_show], label='Actual', alpha=0.7)
    axes[1].plot(pred[:n_show], label='Predicted', alpha=0.7)
    axes[1].set_xlabel('Frame'); axes[1].set_ylabel(f'log({name}) ({unit})'); axes[1].legend()
    axes[1].set_title(f'Predicted vs Actual {name} (time series)')
    plt.tight_layout(); plt.show()

In [ ]:
results_of, results_of_shuffle = fit_target("OF_stim", population_ridge_decoder.of_decoder)
trials_df['OF_pred'] = results_of['y_pred_trials']; trials_df['OF_actual_log'] = results_of['y_test_trials']
print(f'OF: R² = {results_of["r2"]:.3f} (shuffle {results_of_shuffle["r2"]:.3f})')
plot_target(results_of, "OF", "rad/s")

In [ ]:
results_rs, results_rs_shuffle = fit_target("RS_stim", population_ridge_decoder.rs_decoder)
trials_df['RS_pred'] = results_rs['y_pred_trials']; trials_df['RS_actual_log'] = results_rs['y_test_trials']
print(f'RS: R² = {results_rs["r2"]:.3f} (shuffle {results_rs_shuffle["r2"]:.3f})')
plot_target(results_rs, "RS", "m/s")

In [ ]:
results_depth, results_depth_shuffle = fit_target("depth", population_ridge_decoder.depth_decoder)
trials_df['depth_pred'] = results_depth['y_pred_trials']; trials_df['depth_actual_log'] = results_depth['y_test_trials']
print(f'Depth: R² = {results_depth["r2"]:.3f} (shuffle {results_depth_shuffle["r2"]:.3f})')
plot_target(results_depth, "depth", "m")

### The depth-orthogonal RS×OF decoder

`rsof_product_decoder` targets `rsof_product_stim` = RS·OF, the axis orthogonal to depth in
the (log RS, log OF) plane.

In [ ]:
results_rsof, results_rsof_shuffle = fit_target("rsof_product_stim", population_ridge_decoder.rsof_product_decoder)
trials_df['rsof_pred'] = results_rsof['y_pred_trials']; trials_df['rsof_actual_log'] = results_rsof['y_test_trials']
print(f'RS×OF: R² = {results_rsof["r2"]:.3f} (shuffle {results_rsof_shuffle["r2"]:.3f})')
plot_target(results_rsof, "RS×OF", "m·rad/s²")

In [ ]:
print(f'OF    decoder: R² = {results_of["r2"]:.3f}, r = {results_of["pearson_r"]:.3f}')
print(f'RS    decoder: R² = {results_rs["r2"]:.3f}, r = {results_rs["pearson_r"]:.3f}')
print(f'Depth decoder: R² = {results_depth["r2"]:.3f}, r = {results_depth["pearson_r"]:.3f}')
print(f'RS×OF decoder: R² = {results_rsof["r2"]:.3f}, r = {results_rsof["pearson_r"]:.3f}')
print(f'(shuffle) OF {results_of_shuffle["r2"]:.3f} | RS {results_rs_shuffle["r2"]:.3f} | '
      f'Depth {results_depth_shuffle["r2"]:.3f} | RS×OF {results_rsof_shuffle["r2"]:.3f}')

## 3. Decoder performance vs population size (example session)

How many neurons are needed to decode each target? We resample neuron subsets of
increasing size and re-fit each decoder.

In [ ]:
subset_sizes = [5, 10, 20, 50, 75, 100, 200, 400, 600, "inf"]
sub_kwargs = dict(trials_df=trials_df, subset_sizes=subset_sizes, n_resamples="auto",
                  closed_loop=1, rolling_window=None, downsample_window=None,
                  frame_rate=fs, log_transform=True)

print("Subsampling OF...")
sub_of = population_ridge_decoder.decode_with_neuron_subsets(decoder_func=population_ridge_decoder.of_decoder, **sub_kwargs)
print("Subsampling RS...")
sub_rs = population_ridge_decoder.decode_with_neuron_subsets(decoder_func=population_ridge_decoder.rs_decoder, **sub_kwargs)
print("Subsampling Depth...")
sub_depth = population_ridge_decoder.decode_with_neuron_subsets(decoder_func=population_ridge_decoder.depth_decoder, **sub_kwargs)
print("Subsampling RS×OF...")
sub_rsof = population_ridge_decoder.decode_with_neuron_subsets(decoder_func=population_ridge_decoder.rsof_product_decoder, **sub_kwargs)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
panels = [("Optic Flow (OF)", sub_of, "tab:blue"),
          ("Running Speed (RS)", sub_rs, "tab:orange"),
          ("Virtual Depth", sub_depth, "tab:green"),
          ("RS×OF (depth-orthogonal)", sub_rsof, "tab:red")]

for i, (ax, (title, res, color)) in enumerate(zip(axes, panels)):
    sizes = res["subset_sizes"]; r2_means = res["r2_mean"]; r2_stds = res["r2_std"]
    sems = [s / np.sqrt(5) for s in r2_stds[:-1]]
    ax.errorbar(sizes[:-1], r2_means[:-1], yerr=sems, fmt='o-', color=color,
                capsize=5, lw=2, elinewidth=1.5, label='Subsets (mean ± SEM)')
    ax.plot(sizes[-1], r2_means[-1], 's', color='black', markersize=8, label='Full population')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.7)
    ax.set_xlabel('Number of Neurons'); ax.set_title(title); ax.grid(True, alpha=0.3)
    if i == 0: ax.set_ylabel('Decoder R²')
    ax.legend(loc='lower right', fontsize=9)
plt.suptitle('Decoder Performance vs Neuron Population Size (example session)', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### Subset performance separated by layer (across sessions)

The example session above is a single recording depth (L2/3), so layer differences only show
up **across sessions**. Here we load the cut-treadmill neuron-subset results for all sessions
and compare Layer 2/3 vs Layer 5 for each target (each session's full-population point
excluded).

In [ ]:
LAYER_TARGETS = [("OF_stim", "Optic Flow (OF)"), ("RS_stim", "Running Speed (RS)"),
                 ("depth", "Depth"), ("rsof_product_stim", "RS×OF (depth-orthogonal)")]
CUT_OFF = 350
layer_colors = {"Layer 2/3": "#1a5276", "Layer 5": "#b03a2e"}

dfs = []
for project in ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]:
    df = load_project_subsets(project, session_to_exclude=SESSIONS_TO_EXCLUDE.keys(), cut_treadmill=True)
    if not df.empty:
        df["project"] = project
        dfs.append(df)
subsets_all_df = pd.concat(dfs, ignore_index=True)

# Exclude each session's largest subset size (the full-population point)
max_sizes = subsets_all_df.groupby("session")["subset_size"].transform("max")
filtered_subsets = subsets_all_df[subsets_all_df["subset_size"] < max_sizes]

fig, axes = plt.subplots(1, len(LAYER_TARGETS), figsize=(5.2 * len(LAYER_TARGETS), 5), sharex=True, sharey=True)
for ax, (col, label) in zip(axes, LAYER_TARGETS):
    cond_df = filtered_subsets[(filtered_subsets["target"] == col) & (filtered_subsets["condition"] == "closedloop")]
    for layer, marker, ls in [("Layer 2/3", "o", "-"), ("Layer 5", "s", "--")]:
        ld = cond_df[cond_df.nominal_depth <= CUT_OFF] if layer == "Layer 2/3" else cond_df[cond_df.nominal_depth > CUT_OFF]
        if ld.empty:
            continue
        g = ld.groupby("subset_size")["r2_mean"]
        m, s = g.mean(), g.sem()
        color = layer_colors[layer]
        ax.plot(m.index, m.values, marker=marker, linestyle=ls, color=color, linewidth=2, label=layer)
        ax.fill_between(m.index, m.values - s.values, m.values + s.values, color=color, alpha=0.12)
    ax.set_xscale("log"); ax.set_title(label); ax.set_xlabel("Number of Neurons")
    ax.grid(True, alpha=0.3)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
axes[0].set_ylabel("Decoder R²"); axes[0].legend(loc="lower right", fontsize=9)
plt.suptitle("Decoder R² vs neuron subset size by layer (across sessions, cut treadmill)",
             y=1.02, fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## 4. Decoder R² across trial time (example session)

Using the example session's cross-validated predictions, we look at how decoder R² evolves
across the trial, grouped by motor speed. This shows whether the decoder is more accurate in
steady-state vs early in the trial.

In [ ]:
def compute_timepoint_r2(pred_arrays, true_arrays, n_bins=100):
    """R² at each normalized-time bin across a group of trials."""
    time_frac = np.linspace(0, 1, n_bins)
    pred_i = np.full((len(pred_arrays), n_bins), np.nan)
    true_i = np.full((len(true_arrays), n_bins), np.nan)
    for i, (p, t) in enumerate(zip(pred_arrays, true_arrays)):
        p = np.asarray(p).ravel(); t = np.asarray(t).ravel()
        if len(p) < 2 or len(t) < 2:
            continue
        tt = np.linspace(0, 1, len(p))
        pred_i[i] = interp1d(tt, p, kind="linear", fill_value="extrapolate")(time_frac)
        true_i[i] = interp1d(tt, t, kind="linear", fill_value="extrapolate")(time_frac)
    r2 = np.full(n_bins, np.nan); n_valid = np.zeros(n_bins, dtype=int)
    for j in range(n_bins):
        valid = ~(np.isnan(pred_i[:, j]) | np.isnan(true_i[:, j]))
        n_valid[j] = valid.sum()
        if n_valid[j] < 3:
            continue
        pv, tv = pred_i[valid, j], true_i[valid, j]
        ss_tot = np.sum((tv - tv.mean()) ** 2)
        if ss_tot > 0:
            r2[j] = 1 - np.sum((tv - pv) ** 2) / ss_tot
    return time_frac, r2, n_valid

In [ ]:
trials_df['expected_RS'] = trials_df.MotorSpeed_stim.map(np.nanmedian)
cl = trials_df[trials_df.closed_loop == 1]
all_speeds = sorted(s for s in cl['expected_RS'].dropna().unique() if s > 0)

TARGETS = [("RS_stim", "RS_pred", "RS_actual_log", "Running Speed (RS)"),
           ("OF_stim", "OF_pred", "OF_actual_log", "Optic Flow (OF)"),
           ("depth", "depth_pred", "depth_actual_log", "Depth"),
           ("rsof", "rsof_pred", "rsof_actual_log", "RS×OF (depth-orthogonal)")]

speed_norm = plt.Normalize(vmin=min(all_speeds), vmax=max(all_speeds))
speed_cmap = plt.cm.plasma

fig, axes = plt.subplots(1, 4, figsize=(22, 5), sharey=True)
for ax, (key, pred_col, true_col, label) in zip(axes, TARGETS):
    for speed in all_speeds:
        sub = cl[(cl['expected_RS'] == speed) & cl[pred_col].notna() & cl[true_col].notna()]
        if len(sub) < 5:
            continue
        tf, r2, nv = compute_timepoint_r2(sub[pred_col].tolist(), sub[true_col].tolist())
        ax.plot(tf, r2, color=speed_cmap(speed_norm(speed)), linewidth=2, label=f"{speed:.0f} cm/s")
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_xlabel("Normalized trial time"); ax.set_title(label)
axes[0].set_ylabel("R²"); axes[0].legend(fontsize=8, ncol=2, framealpha=0.8)
plt.suptitle(f"Decoder R² across trial time by motor speed ({EXAMPLE_SESSION})", y=1.02, fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()